# Multi-Agent Collaboration

worth mentioning: https://princeton-nlp.github.io/SocraticAI/

adapt this lab from: https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/examples/travel-planning.html

In [1]:
# !uv pip install -U "autogen-agentchat" "autogen-ext[ollama]"

In [1]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
# from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.models.ollama import OllamaChatCompletionClient
from autogen_core.models import ModelInfo


### RoundRobinGroupChat

In [2]:
model_client = OllamaChatCompletionClient(
    model="scb10x/typhoon2.5-qwen3-4b",
    model_info = ModelInfo(vision=False, function_calling=True, json_output=True, family="qwen", structured_output=True)
)

planner_agent = AssistantAgent(
    "planner_agent",
    model_client=model_client,
    description="A helpful assistant that can plan trips.",
    system_message="You are a helpful assistant that can suggest a travel plan for a user based on their request.",
)

local_agent = AssistantAgent(
    "local_agent",
    model_client=model_client,
    description="A local assistant that can suggest local activities or places to visit.",
    system_message="You are a helpful assistant that can suggest authentic and interesting local activities or places to visit for a user and can utilize any context information provided.",
)

language_agent = AssistantAgent(
    "language_agent",
    model_client=model_client,
    description="A helpful assistant that can provide language tips for a given destination.",
    system_message="You are a helpful assistant that can review travel plans, providing feedback on important/critical tips about how best to address language or communication challenges for the given destination. If the plan already includes language tips, you can mention that the plan is satisfactory, with rationale.",
)

travel_summary_agent = AssistantAgent(
    "travel_summary_agent",
    model_client=model_client,
    description="A helpful assistant that can summarize the travel plan.",
    system_message="You are a helpful assistant that can take in all of the suggestions and advice from the other agents and provide a detailed final travel plan. You must ensure that the final plan is integrated and complete. YOUR FINAL RESPONSE MUST BE THE COMPLETE PLAN. When the plan is complete and all perspectives are integrated, you can respond with TERMINATE.",
)


In [3]:
termination = TextMentionTermination("TERMINATE")
group_chat = RoundRobinGroupChat(
    [planner_agent, local_agent, language_agent, travel_summary_agent], 
    termination_condition=termination,
    max_turns=5
)
await Console(group_chat.run_stream(task="Plan a 3 day trip to Phuket."))

await model_client.close()


---------- TextMessage (user) ----------
Plan a 3 day trip to Phuket.
---------- TextMessage (planner_agent) ----------
Sure! Here's a well-balanced 3-day itinerary for your Phuket trip:

**Day 1: Arrival & Beach Relaxation**
- Arrive at Phuket International Airport and transfer to your hotel (preferably near Patong Beach or Karon Beach).
- Check in and freshen up.
- Spend the afternoon relaxing at Patong Beach—great for sunbathing, swimming, and people-watching.
- Optional: Try a local seafood dinner at a beachfront restaurant like Seafood House or The Seafood Bar.
- Evening: Take a short walk along the beach and enjoy sunset views.

**Day 2: Island Hopping & Cultural Exploration**
- Morning: Take a boat trip to nearby islands like Phi Phi Islands or Koh Phi Phi (book in advance). Enjoy snorkeling or swimming in crystal-clear waters.
- Afternoon: Visit the vibrant night markets at Patong or explore the quieter charm of Old Phuket Town.
- Stop by the Phuket History Museum or Wat Chalon

In [4]:
"DONE"

'DONE'

### SelectorGroupChat

In [5]:
model_client = OllamaChatCompletionClient(
    model="scb10x/typhoon2.5-qwen3-4b",
    model_info = ModelInfo(vision=False, function_calling=True, json_output=True, family="qwen", structured_output=True)
)

planner_agent = AssistantAgent(
    "planner_agent",
    model_client=model_client,
    description="A helpful assistant that can plan trips.",
    system_message="You are a helpful assistant that can suggest a travel plan for a user based on their request.",
)

local_agent = AssistantAgent(
    "local_agent",
    model_client=model_client,
    description="A local assistant that can suggest local activities or places to visit.",
    system_message="You are a helpful assistant that can suggest authentic and interesting local activities or places to visit for a user and can utilize any context information provided.",
)

language_agent = AssistantAgent(
    "language_agent",
    model_client=model_client,
    description="A helpful assistant that can provide language tips for a given destination.",
    system_message="You are a helpful assistant that can review travel plans, providing feedback on important/critical tips about how best to address language or communication challenges for the given destination. If the plan already includes language tips, you can mention that the plan is satisfactory, with rationale.",
)

travel_summary_agent = AssistantAgent(
    "travel_summary_agent",
    model_client=model_client,
    description="A helpful assistant that can summarize the travel plan.",
    system_message="You are a helpful assistant that can take in all of the suggestions and advice from the other agents and provide a detailed final travel plan. You must ensure that the final plan is integrated and complete. YOUR FINAL RESPONSE MUST BE THE COMPLETE PLAN. When the plan is complete and all perspectives are integrated, you can respond with TERMINATE.",
)

termination = TextMentionTermination("TERMINATE")

In [ ]:
from autogen_agentchat.teams import SelectorGroupChat

group_chat = SelectorGroupChat(
    [planner_agent, local_agent, language_agent, travel_summary_agent], 
    model_client=model_client,
    termination_condition=termination,
    allow_repeated_speaker=True,  # Allow an agent to speak multiple turns in a row.
    max_turns=5,
)
await Console(group_chat.run_stream(task="Plan a 3 day trip to Phuket."))

await model_client.close()

---------- TextMessage (user) ----------
Plan a 3 day trip to Phuket.
---------- TextMessage (planner_agent) ----------
Sure! Here's a well-balanced 3-day itinerary for your trip to Phuket:

**Day 1: Arrival & Beach Relaxation**

- **Afternoon**: Arrive at Phuket International Airport. Transfer to your hotel (recommendation: stay in Patong Beach or Karon Beach for easy access and amenities).
- **Evening**: Head to Patong Beach for a relaxed stroll along the shore. Enjoy sunset views, grab a drink at one of the beachside bars, or unwind at a beachside restaurant.
- **Dinner**: Try local seafood at a seaside restaurant like “Pattaya Seafood” or “Siam Seafood Restaurant.”

**Day 2: Island Hopping & Cultural Exploration**

- **Morning**: Take a boat tour to nearby islands such as Koh Phi Phi or Koh Panyi. You can enjoy snorkeling, swimming, and scenic views.
- **Afternoon**: Visit the Phuket Cultural Village to experience local Thai traditions, crafts, and food. Try traditional Thai dishes

In [ ]:
"DONE"